In [ ]:
import os, glob, json, sqlite3
import pandas as pd

# DATA_DIR is the search output directory (contains campaign.sqlite). The analysis
# runner injects it; fall back to the current directory.
DATA_DIR = ''
db = os.path.join(DATA_DIR, 'campaign.sqlite')
if not os.path.exists(db):
    hits = glob.glob(os.path.join(DATA_DIR or '.', '**', 'campaign.sqlite'), recursive=True)
    db = hits[0] if hits else db

conn = sqlite3.connect(db)
rows = conn.execute('SELECT params_json, objectives_json, measures_json, n_samples FROM unit').fetchall()
conn.close()

records = []
for params_json, objectives_json, measures_json, n in rows:
    rec = {}
    rec.update(json.loads(params_json or '{}'))
    rec.update(json.loads(objectives_json or '{}'))
    rec.update(json.loads(measures_json or '{}'))
    rec['n_samples'] = n
    records.append(rec)
df = pd.DataFrame(records)
print(f'{len(df)} evaluated parameter set(s)')
df.head()

In [ ]:
import itertools
import matplotlib.pyplot as plt

# Objective distribution (e.g. failure_rate across all evaluated configs).
obj_cols = [c for c in df.columns if c in ('failure_rate',)] or [df.columns[0]]
obj = obj_cols[0]
fig, ax = plt.subplots(figsize=(6, 3))
df[obj].hist(bins=20, ax=ax)
ax.set_xlabel(obj); ax.set_ylabel('configs'); ax.set_title(f'{obj} distribution')
plt.tight_layout(); plt.show()

In [ ]:
# Behavior space: 2-D projections of every measure pair, coloured by objective.
# For QD this is the discovered failure landscape; for random/optuna it is the
# evaluated cloud.
measures = [c for c in ['max_tilt', 'drift_dist', 'landing_speed', 'control_effort'] if c in df.columns]
pairs = list(itertools.combinations(measures, 2))
if pairs:
    fig, axes = plt.subplots(1, len(pairs), figsize=(5 * len(pairs), 4), squeeze=False)
    for ax, (mx, my) in zip(axes[0], pairs):
        sc = ax.scatter(df[mx], df[my], c=df[obj], cmap='viridis')
        ax.set_xlabel(mx); ax.set_ylabel(my)
        fig.colorbar(sc, ax=ax, label=obj)
    fig.suptitle('Behavior space coloured by ' + obj)
    plt.tight_layout(); plt.show()
print('distinct (rounded) behavior cells covered:',
      df[measures].round(2).drop_duplicates().shape[0] if measures else 0)